In [1]:
import json
import pandas as pd

In [ ]:
with open("outputs/code_switch_metadata.json", "r") as f:
    metadata = json.load(f)

In [3]:
rows = []

for split, items in metadata.items():
    for item in items:
        rows.append({
            "split": split,
            "index": item["index"],
            "speaker_id": item["speaker_id"],
            "session_id": item["session_id"],
            "duration": item["duration"],
            "num_switches": item["num_switches"],
            "switches": item["switches"],
            "inter_switch_intervals": item["inter_switch_intervals"]
        })

df = pd.DataFrame(rows)

In [4]:
q25_duration = df["duration"].quantile(0.25)
q75_duration = df["duration"].quantile(0.75)

print("Duration thresholds:")
print("Q25:", q25_duration)
print("Q75:", q75_duration)

Duration thresholds:
Q25: 2.6
Q75: 6.0


In [5]:
def duration_category(duration):
    if duration <= q25_duration:
        return "short"
    elif duration <= q75_duration:
        return "medium"
    else:
        return "long"

In [6]:
def position_category(relative_position):
    if relative_position <= 1/3:
        return "early"
    elif relative_position <= 2/3:
        return "middle"
    else:
        return "late"

In [7]:
interval_rows = []

for _, row in df.iterrows():
    for interval in row["inter_switch_intervals"]:
        interval_rows.append({
            "interval": interval
        })

interval_df = pd.DataFrame(interval_rows)

In [8]:
q25_interval = interval_df["interval"].quantile(0.25)
q50_interval = interval_df["interval"].quantile(0.50)
q75_interval = interval_df["interval"].quantile(0.75)

print("Interval thresholds:")
print("Q25:", q25_interval)
print("Q50:", q50_interval)
print("Q75:", q75_interval)

Interval thresholds:
Q25: 0.401
Q50: 0.583
Q75: 0.985


In [9]:
def interval_category(interval):
    if interval <= q25_interval:
        return "close"
    elif interval <= q50_interval:
        return "medium"
    elif interval <= q75_interval:
        return "far"
    else:
        return "very_far"

In [10]:
coverage = {}

for split, items in metadata.items():

    for item in items:
        
        idx = (split, item["index"])
       
        categories = {
            f"speaker:{item['speaker_id']}",
            f"session:{item['session_id']}",
            f"duration:{duration_category(item['duration'])}",
            f"num_switches:{item['num_switches']}",
        }
        
        for switch in item["switches"]:

            categories.add(f"direction:{switch['direction']}")
            categories.add(f"position:{position_category(switch['relative_position'])}")

        for interval in item["inter_switch_intervals"]:

            categories.add(f"interval:{interval_category(interval)}")

        coverage[idx] = categories

In [11]:
required_categories = set().union(*coverage.values())


In [12]:
print("Total utterances:", len(coverage))
print("Total categories to cover:", len(required_categories))

Total utterances: 3118
Total categories to cover: 52


In [13]:
selected = []
covered = set()
remaining = list(coverage.keys())

while covered != required_categories:

    best_idx = None
    best_gain = set()

    for idx in remaining:

        gain = coverage[idx] - covered

        if len(gain) > len(best_gain):
            best_idx = idx
            best_gain = gain

    if best_idx is None:
        raise RuntimeError("Could not cover all categories.")

    selected.append(best_idx)
    covered.update(coverage[best_idx])
    remaining.remove(best_idx)

    print(f"Selected {len(selected)}: {best_idx} | new categories = {len(best_gain)} | covered = {len(covered)}/{len(required_categories)}")

Selected 1: ('train', 1040) | new categories = 13 | covered = 13/52
Selected 2: ('train', 188) | new categories = 4 | covered = 17/52
Selected 3: ('train', 499) | new categories = 4 | covered = 21/52
Selected 4: ('train', 785) | new categories = 3 | covered = 24/52
Selected 5: ('train', 107) | new categories = 2 | covered = 26/52
Selected 6: ('train', 293) | new categories = 2 | covered = 28/52
Selected 7: ('train', 1152) | new categories = 2 | covered = 30/52
Selected 8: ('train', 3529) | new categories = 2 | covered = 32/52
Selected 9: ('train', 5451) | new categories = 2 | covered = 34/52
Selected 10: ('train', 5728) | new categories = 2 | covered = 36/52
Selected 11: ('train', 7522) | new categories = 2 | covered = 38/52
Selected 12: ('test', 402) | new categories = 2 | covered = 40/52
Selected 13: ('train', 668) | new categories = 1 | covered = 41/52
Selected 14: ('train', 1155) | new categories = 1 | covered = 42/52
Selected 15: ('train', 1313) | new categories = 1 | covered = 43

In [ ]:
coverage_subset = []

for split, idx in selected:

    coverage_subset.append({
        "split": split,
        "index": idx
    })

with open("outputs/coverage_subset.json", "w") as f:
    json.dump(coverage_subset, f, indent=2)

In [15]:
print("Selected utterances:", len(selected))
print("Categories covered:", len(covered), "/", len(required_categories))

missing = required_categories - covered

if missing:
    print("Missing categories:", missing)
else:
    print("All categories are covered.")

Selected utterances: 24
Categories covered: 52 / 52
All categories are covered.
